# Fleet and Egress load

To support server placement planning and
load balancing (adjusting server probabilities).

Since the project's inception all egress has been provided by in-kind contributions
by Google or other ISPs. Measurement Lab (M-Lab) has never directly paid for network egress traffic.

This dashboard estimates the market value of these in-kind contributions,
assuming a flat rate of $0.10 per GB.  This overestimates the value in North American and Europe and
under estimates the value in most of the rest of the world.

A future version will support additional, more authentic, pricing models.

For definitions of the notation and other information see the description [In-Kind Egress Contributions to M-Lab](https://docs.google.com/document/d/1AGxZ8mvM8H7ehHxhE7CpNycRGjnCsEnJnHW0pHd2ErM/edit?usp=sharing).

In [ ]:
# --- Setup ---
import os, sys, json
from datetime import datetime, date, time, timedelta, timezone

# Locate the repo root (directory containing `converter/`) regardless of where
# Voila/Jupyter is launched from.
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "converter")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

import ipywidgets as widgets
import plotly.graph_objects as go
import pandas as pd
from IPython.display import display, HTML, Markdown

from converter import query_builder as qb, runtime as rt
from converter.widget_builder import Controls

client = rt.bq_client()

# Dashboard variable metadata baked in at conversion time.
VARIABLES = json.loads(r"""
[
  {
    "name": "costModel",
    "type": "custom",
    "label": "Value Model",
    "description": "Regional per SKU costs used to generate overall costs",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "Flat $0.10 per Gigabyte", "value": "Flat $0.10 per Gigabyte" }
    ],
    "current": { "value": "Flat $0.10 per Gigabyte" },
    "query_sql": "Flat $0.10 per Gigabyte"
  },
  {
    "name": "duration",
    "type": "custom",
    "label": "",
    "description": "Number of days to include in the sampel",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "1", "value": "1" },
      { "text": "7", "value": "7" },
      { "text": "30", "value": "30" }
    ],
    "current": { "value": "1" },
    "query_sql": "1, 7, 30"
  },
  {
    "name": "endDate",
    "type": "textbox",
    "label": "",
    "description": "Sample end",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "2026-04-20", "value": "2026-04-20" }
    ],
    "current": { "value": "2026-04-20" },
    "query_sql": "2026-04-20",
    "dynamic_default": "(date.today() - timedelta(days=2)).isoformat()"
  },
  {
    "name": "display",
    "type": "custom",
    "label": "Level of detail",
    "description": "Select between site details and site summaries.",
    "hide": 0,
    "multi": true,
    "options": [
      { "text": "summaries", "value": "summaries" },
      { "text": "metros", "value": "metros" },
      { "text": "sites", "value": "sites" }
    ],
    "current": {
      "value": [
        "metros"
      ]
    },
    "query_sql": "summaries, metros, sites"
  }
]
""")


In [ ]:
# --- URL parameter presets (webapp mode) ---
# Voila injects the request query string into os.environ["QUERY_STRING"] before
# executing the notebook.  get_query_string() also handles the preheat-kernel
# case (blocks until the request arrives).  Falls back gracefully in plain
# Jupyter where neither is set.
# For scripted or test overrides, set DASH_PRESETS to a JSON object.
import urllib.parse

url_params = {}
try:
    from voila.utils import get_query_string
    _qs = get_query_string() or ""
    for _k, _vs in urllib.parse.parse_qs(_qs).items():
        url_params[_k] = _vs[0] if len(_vs) == 1 else _vs
except Exception:
    pass

_env = os.environ.get("DASH_PRESETS")
if _env:
    url_params.update(json.loads(_env))


In [ ]:
# --- Dashboard controls (dropdowns; query-backed ones are chained) ---
ctrl = Controls(VARIABLES, client, presets=url_params)
w_run = widgets.Button(description="Run / Refresh", button_style="primary", icon="play")

w_from = w_to = None
_date_label = widgets.HTML('')   # filled from query results after Run


In [ ]:
# --- Panels (converted from the dashboard) ---
SUMMARY_PANELS  = [
  {
    'id': 1,
    'title': 'Server egress value estimated from TCPinfo using ${costModel}  for ${duration} days ending ${endDate}',
    'type': 'table',
    'sql': r"""
SELECT -- reorder fields to make it prettier
  * EXCEPT (lat, long),
  lat, long
FROM `mm_preproduction.global_fleet_inventory`(DATE_SUB("$endDate", INTERVAL ${duration:raw}-1 DAY), DATE("$endDate"))
WHERE
  ("${display}" like '%summar%' AND tag LIKE '!%') OR
  ("${display}" like '%metro%' AND level LIKE '%Metro%') OR
  ("${display}" like '%site%' AND level LIKE '%Site%')
""".strip(),
    'layout': {},
    'skip_if_field_none': False,
  },
]
METRIC_LAYOUTS  = json.loads(r"""{
  "MeanThroughputMbps": {
    "type": "log",
    "autorange": false,
    "range": [
      -0.3,
      3.3
    ],
    "gridcolor": "#333"
  },
  "MinRTT": {
    "type": "log",
    "autorange": false,
    "range": [
      -0.3,
      3.0
    ],
    "gridcolor": "#333"
  },
  "linearMinRTT": {
    "type": "linear",
    "autorange": false,
    "range": [
      0,
      300
    ],
    "gridcolor": "#333"
  },
  "LossRate": {
    "type": "log",
    "autorange": true,
    "gridcolor": "#333"
  }
}""")
REPEAT_VAR      = None

out = widgets.Output()


def _diagnostics(ctx):
    rows = [(k, ", ".join(v) if isinstance(v, list) else str(v))
            for k, v in ctx.items()]
    return pd.DataFrame(rows, columns=["variable", "value"])


def render(_=None):
    ctx = ctrl.context()
    to_dt   = (datetime.combine(w_to.value,   time(), tzinfo=timezone.utc)
               if w_to   and w_to.value   else datetime.now(timezone.utc))
    from_dt = (datetime.combine(w_from.value, time(), tzinfo=timezone.utc)
               if w_from and w_from.value else to_dt - timedelta(days=7))
    cache = {}

    def query(sql):
        if sql not in cache:
            cache[sql] = rt.run_query(client, sql)
        return cache[sql]

    out.clear_output(wait=True)
    with out:
        # Default to "Summary" when table_style is absent (e.g. fleet dashboard).
        table_style = ctx.get("table_style",
                               "Summary" if SUMMARY_PANELS else "none")
        if table_style != "none":
            _sctx = dict(ctx)
            if "table_style" in ctx:
                # Prod: map table_style → verbose flag expected by regional_report SQL.
                _sctx["verbose"] = "true" if table_style == "Verbose" else "false"
            # Other flavors (barchart, fleet) pass verbose directly from ctx.
            for p in SUMMARY_PANELS:
                display(Markdown("### " + qb.interpolate(p["title"], ctx)))
                sql = qb.interpolate(p["sql"], _sctx, from_dt=from_dt, to_dt=to_dt)
                try:
                    _df = query(sql)
                except Exception as exc:
                    display(HTML(f"<pre>query failed: {exc}</pre>"))
                    _df = None
                if _df is not None:
                    if p.get("type") == "barchart":
                        _fig = rt.metro_barchart(
                            _df, isp_count=ctx.get("ISPcount", "5"))
                        _fig._config = {"responsive": False}
                        display(_fig)
                        display(HTML(rt.metro_nav_html(
                            _df, isp_count=ctx.get("ISPcount", "5"))))
                    else:
                        _display_sel = ctx.get("display") or []
                        if isinstance(_display_sel, str): _display_sel = [_display_sel]
                        if any(d in _display_sel for d in ("metros", "sites")):
                            _map_df = _df[_df["lat"].notna() & _df["long"].notna()].copy()
                            if not _map_df.empty:
                                display(go.FigureWidget(rt.fleet_map(_map_df)))
                        display(HTML(
                            '<div style="height:500px;overflow:auto">'
                            + _df.to_html(index=False, na_rep="")
                            + '</div>'
                        ))

        repeats = ctx.get(REPEAT_VAR) or []
        if isinstance(repeats, str):
            repeats = [repeats]

        selected_metrics = ctx.get("metrics") or []
        if isinstance(selected_metrics, str):
            selected_metrics = [selected_metrics]

        # One Python call per selected metric fetches data for all client ISPs.
        bulk_by_metric = {}
        _site_regex = qb.format_regex(ctx.get("region") or [])
        _isp_regex  = rt.asn_regex(repeats)
        for metric in selected_metrics:
            try:
                bulk_by_metric[metric] = rt.fetch_histograms(
                    client,
                    method=ctx.get("method", "cached"),
                    field=metric,
                    site_regex=_site_regex,
                    isp_count=int(ctx.get("ISPcount", 10)),
                    bin_size=int(ctx.get("binSize", 50)),
                    x_axis=ctx.get("xAxis", "none"),
                    from_dt=from_dt,
                    to_dt=to_dt,
                    isp_regex=_isp_regex,
                    dataset=ctx.get("dataset",
                                    "mlab-collaboration.mm_preproduction"),
                )
            except Exception as exc:
                bulk_by_metric[metric] = exc

        # Update cached date range label from metroStart/metroEnd in query results.
        for _mdf in bulk_by_metric.values():
            if isinstance(_mdf, pd.DataFrame) and 'metroStart' in _mdf.columns:
                _s = pd.to_datetime(_mdf['metroStart'].dropna().min()).date()
                _e = pd.to_datetime(_mdf['metroEnd'].dropna().max()).date()
                _date_label.value = (
                    '<div style="font-size:12px;color:grey;margin:2px 0">'
                    '<b>Cached data:</b> ' + str(_s) + ' – ' + str(_e) + '</div>')
                break

        for value in repeats:
            asn = str(value).split()[0]
            display(HTML(f"<h3>{REPEAT_VAR}: {value}</h3>"))
            figs = []
            for metric in selected_metrics:
                df_all = bulk_by_metric.get(metric)
                if isinstance(df_all, Exception):
                    figs.append(widgets.HTML(f"<b>{metric}</b><pre>{df_all}</pre>"))
                    continue
                df = df_all[df_all["ISPname"].str.startswith(asn + " ")]
                try:
                    fig = rt.plotly_combined_figure(
                        df, {"xaxis": METRIC_LAYOUTS.get(metric, {})}, title=metric)
                    figs.append(go.FigureWidget(fig))
                except Exception as exc:
                    figs.append(widgets.HTML(f"<b>{metric}</b><pre>{exc}</pre>"))
            if figs:
                display(widgets.HBox(figs, layout=widgets.Layout(flex_flow="row wrap")))

        _diag_out = widgets.Output()
        with _diag_out:
            display(_diagnostics(ctx))
        _diag_acc = widgets.Accordion(children=[_diag_out])
        _diag_acc.set_title(0, 'Selector Diagnostics')
        _diag_acc.selected_index = None   # collapsed by default
        display(_diag_acc)


w_run.on_click(render)


In [ ]:
# --- Display the app ---
_date_row = (widgets.HBox([w_from, w_to],
                          layout=widgets.Layout(margin='2px 0'))
             if w_from is not None else widgets.HTML(''))
display(widgets.VBox([ctrl.box, _date_label, _date_row, w_run, out]))
